<a href="https://colab.research.google.com/github/hilduAgl/brain-age-eeg/blob/main/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Evaluation and Error Analysis

This notebook compares the feature-based linear regression
baseline and the convolutional neural network (CNN) for
predicting chronological age from EEG recordings.

## Objectives
- Compare validation MAE and RMSE
- Visualize predicted versus actual age
- Investigate systematic prediction bias
- Evaluate performance across age groups
- Identify strengths and limitations of each model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path("/content/drive/MyDrive/BrainAge")
RESULTS_DIR = PROJECT_ROOT / "results"

baseline = pd.read_csv(
    RESULTS_DIR / "baseline_val_predictions.csv"
)

cnn = pd.read_csv(
    RESULTS_DIR / "cnn_val_predictions.csv"
)

FIGURES_DIR = RESULTS_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Baseline:", baseline.shape)
print("CNN:", cnn.shape)

display(baseline.head())
display(cnn.head())

In [ ]:
required = {"filename", "actual_age", "predicted_age"}

for name, df in [("Baseline", baseline), ("CNN", cnn)]:
    assert required.issubset(df.columns), f"{name}: missing columns"
    assert df["filename"].is_unique, f"{name}: duplicate subjects"
    assert df[list(required)].notna().all().all()

comparison = baseline.merge(
    cnn,
    on="filename",
    suffixes=("_baseline", "_cnn"),
    validate="one_to_one"
)

assert len(baseline) == len(cnn) == len(comparison) == 103

assert np.allclose(
    comparison["actual_age_baseline"],
    comparison["actual_age_cnn"]
)

comparison["actual_age"] = comparison["actual_age_baseline"]

display(comparison.head())

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Bias": np.mean(np.asarray(y_pred) - np.asarray(y_true))
    }

y_true = comparison["actual_age"]

results = pd.DataFrame({
    "Linear Regression": evaluate(
        y_true, comparison["predicted_age_baseline"]
    ),
    "CNN": evaluate(
        y_true, comparison["predicted_age_cnn"]
    )
}).T

display(results.round(3))

In [ ]:
plt.figure(figsize=(8, 7))

plt.scatter(
    comparison["actual_age"],
    comparison["predicted_age_baseline"],
    alpha=0.6,
    label="Linear Regression"
)

plt.scatter(
    comparison["actual_age"],
    comparison["predicted_age_cnn"],
    alpha=0.6,
    label="CNN"
)

plt.plot(
    [0, 17],
    [0, 17],
    "k--",
    label="Perfect prediction"
)

plt.xlabel("Actual age (years)")
plt.ylabel("Predicted age (years)")
plt.title("Predicted vs Actual Brain Age — Validation Set")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
comparison["error_baseline"] = (
    comparison["predicted_age_baseline"]
    - comparison["actual_age"]
)

comparison["error_cnn"] = (
    comparison["predicted_age_cnn"]
    - comparison["actual_age"]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    comparison["actual_age"],
    comparison["error_baseline"],
    alpha=0.6,
    label="Linear Regression"
)

plt.scatter(
    comparison["actual_age"],
    comparison["error_cnn"],
    alpha=0.6,
    label="CNN"
)

plt.axhline(0, color="black", linestyle="--")

plt.xlabel("Actual age (years)")
plt.ylabel("Prediction error (years)")
plt.title("Prediction Error vs Actual Age")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
comparison["error_baseline"] = (
    comparison["predicted_age_baseline"]
    - comparison["actual_age"]
)

comparison["error_cnn"] = (
    comparison["predicted_age_cnn"]
    - comparison["actual_age"]
)

plt.figure(figsize=(9, 5))

plt.scatter(
    comparison["actual_age"],
    comparison["error_baseline"],
    alpha=0.6,
    label="Linear Regression"
)

plt.scatter(
    comparison["actual_age"],
    comparison["error_cnn"],
    alpha=0.6,
    label="CNN"
)

plt.axhline(0, color="black", linestyle="--")

plt.xlabel("Actual age (years)")
plt.ylabel("Prediction error (years)")
plt.title("Prediction Error vs Actual Age")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
bins = [0, 2, 5, 10, np.inf]
labels = ["0–2", "2–5", "5–10", "10+"]

comparison["age_group"] = pd.cut(
    comparison["actual_age"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

comparison["abs_error_baseline"] = comparison["error_baseline"].abs()
comparison["abs_error_cnn"] = comparison["error_cnn"].abs()

age_results = comparison.groupby(
    "age_group", observed=True
).agg(
    Subjects=("actual_age", "size"),
    Baseline_MAE=("abs_error_baseline", "mean"),
    CNN_MAE=("abs_error_cnn", "mean")
)

display(age_results.round(2))

In [ ]:
ax = age_results[
    ["Baseline_MAE", "CNN_MAE"]
].plot(
    kind="bar",
    figsize=(9, 5)
)

ax.set_xlabel("Age group (years)")
ax.set_ylabel("Mean Absolute Error (years)")
ax.set_title("Validation MAE by Age Group")
ax.legend(["Linear Regression", "CNN"])

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()